In [2]:
import os
import cv2
import numpy as np
from skimage.metrics import structural_similarity as ssim
from tabulate import tabulate

In [3]:
# =========================
# Utility Functions
# =========================

def load_image(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img.astype(np.float32) / 255.0
    return img


def compute_psnr(img1, img2):
    mse = np.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    return -10 * np.log10(mse)


def compute_ssim(img1, img2):
    return ssim(img1, img2, channel_axis=2, data_range=1.0)

In [4]:
# =========================
# Per-image Evaluation
# =========================

def evaluate_selected_pairs(gt_dir, render_dir, indices):
    results = []

    for idx in indices:
        filename = f"{idx:05d}.png"

        gt_path = os.path.join(gt_dir, filename)
        render_path = os.path.join(render_dir, filename)

        if not os.path.exists(gt_path) or not os.path.exists(render_path):
            continue

        gt = load_image(gt_path)
        pred = load_image(render_path)

        psnr_val = compute_psnr(gt, pred)
        ssim_val = compute_ssim(gt, pred)

        results.append([filename, round(psnr_val, 3), round(ssim_val, 4)])

    print("\n=== Per-Image Metrics (Selected Test Pairs) ===")
    print(tabulate(results, headers=["Image", "PSNR", "SSIM"], tablefmt="grid"))

    return results

In [6]:
# =========================
# Dataset-level Evaluation
# =========================

def evaluate_dataset(gt_dir, render_dir):
    psnr_list = []
    ssim_list = []

    files = sorted(os.listdir(gt_dir))

    for filename in files:
        gt_path = os.path.join(gt_dir, filename)
        render_path = os.path.join(render_dir, filename)

        if not os.path.exists(render_path):
            continue

        gt = load_image(gt_path)
        pred = load_image(render_path)

        psnr_list.append(compute_psnr(gt, pred))
        ssim_list.append(compute_ssim(gt, pred))

    mean_psnr = np.mean(psnr_list)
    mean_ssim = np.mean(ssim_list)

    return mean_psnr, mean_ssim



def evaluate_train_test(train_gt, train_render, test_gt, test_render):
    train_psnr, train_ssim = evaluate_dataset(train_gt, train_render)
    test_psnr, test_ssim = evaluate_dataset(test_gt, test_render)

    table = [
        ["Train", round(train_psnr, 3), round(train_ssim, 4)],
        ["Test", round(test_psnr, 3), round(test_ssim, 4)],
    ]

    print("\n=== Dataset-level Metrics ===")
    print(tabulate(table, headers=["Dataset", "Mean PSNR", "Mean SSIM"], tablefmt="grid"))

    return table

In [8]:
# -------- PATHS --------
BASE = "/Users/arjunmallick/Reconstruction_Projects/Main-Building-gsplat/gsplat-results/Main-Building-gsplat/output_model"

test_gt = os.path.join(BASE, "test/ours_15000/gt")
test_render = os.path.join(BASE, "test/ours_15000/renders")

train_gt = os.path.join(BASE, "train/ours_15000/gt")
train_render = os.path.join(BASE, "train/ours_15000/renders")

# -------- SELECTED TEST PAIRS --------
# Replace with indices you used in report
selected_indices = [0, 10, 13]
img_wise_tables = evaluate_selected_pairs(test_gt, test_render, selected_indices)


=== Per-Image Metrics (Selected Test Pairs) ===
+-----------+--------+--------+
| Image     |   PSNR |   SSIM |
+===========+========+========+
| 00000.png | 20.883 | 0.5999 |
+-----------+--------+--------+
| 00010.png | 19.087 | 0.7609 |
+-----------+--------+--------+
| 00013.png | 22.08  | 0.6661 |
+-----------+--------+--------+


In [9]:
# -------- PATHS --------
BASE = "/Users/arjunmallick/Reconstruction_Projects/Main-Building-gsplat/gsplat-results/Main-Building-gsplat/output_model"

test_gt = os.path.join(BASE, "test/ours_15000/gt")
test_render = os.path.join(BASE, "test/ours_15000/renders")

train_gt = os.path.join(BASE, "train/ours_15000/gt")
train_render = os.path.join(BASE, "train/ours_15000/renders")

evaluate_train_test(train_gt, train_render, test_gt, test_render)


=== Dataset-level Metrics ===
+-----------+-------------+-------------+
| Dataset   |   Mean PSNR |   Mean SSIM |
+===========+=============+=============+
| Train     |      21.014 |      0.6635 |
+-----------+-------------+-------------+
| Test      |      18.658 |      0.6457 |
+-----------+-------------+-------------+


[['Train', 21.014, 0.6635], ['Test', 18.658, 0.6457]]